# 🔬 Lab Module 24: Agent Knowledge & Epistemics

**Mục tiêu:** Implement và test các components giúp LoanBot biết *những gì nó không biết*.

| Section | Topic |
|---------|-------|
| 1 | Knowledge Boundary Classifier |
| 2 | Hallucination Detection (Citation + Numeric) |
| 3 | Self-Consistency Sampling |
| 4 | Calibration & ECE Measurement |
| 5 | RAG Faithfulness Checker |
| 6 | Knowledge Freshness Monitor |
| 7 | Epistemic Guard Orchestrator (Full Pipeline) |

## Section 1: Knowledge Boundary Classifier

Classify queries vào 3 loại: KNOWN_KNOWN / KNOWN_UNKNOWN / UNCERTAIN.

In [ ]:
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
from enum import Enum
import re
import statistics
import hashlib
from datetime import date

class KnowledgeType(Enum):
    KNOWN_KNOWN   = 'known_known'
    KNOWN_UNKNOWN = 'known_unknown'
    UNCERTAIN     = 'uncertain'

@dataclass
class KnowledgeBoundaryResult:
    knowledge_type: KnowledgeType
    confidence: float
    response_strategy: str
    suggested_source: str = ''
    matched_signal: str = ''

KNOWN_DOMAINS = {
    'credit_score_thresholds': ('TT39/2016 thresholds: score<500 reject, 500-649 review, ≥650 approve', 0.95),
    'dti_limits': ('DTI max 0.45 standard per FinTech Corp policy v3.6', 0.92),
    'loan_decision_process': ('5-step underwriting: blacklist→credit→income→employment→collateral', 0.98),
    'blacklist_check': ('Always step 1: CIC query + internal blacklist', 0.99),
    'loan_types': ('Personal loan, mortgage, auto loan, SME loan', 0.90),
}

OUT_OF_DOMAIN_SIGNALS = [
    'lãi suất hôm nay', 'lãi suất hiện tại', 'giá vàng', 'tỷ giá', 'tỷ lệ lạm phát',
    'thông tư mới nhất', 'quy định 2024', 'quy định 2025', 'circular mới',
    'thị trường bất động sản', 'lãi suất tối đa', 'lãi suất cơ bản',
    'interest rate today', 'current interest rate',
]

def classify_knowledge_boundary(query: str) -> KnowledgeBoundaryResult:
    q = query.lower()
    for signal in OUT_OF_DOMAIN_SIGNALS:
        if signal in q:
            return KnowledgeBoundaryResult(
                knowledge_type=KnowledgeType.KNOWN_UNKNOWN,
                confidence=0.15,
                response_strategy='decline_and_redirect',
                suggested_source='sbv.gov.vn hoặc phòng pháp chế',
                matched_signal=signal
            )
    for domain, (_, conf) in KNOWN_DOMAINS.items():
        keywords = domain.split('_')
        if any(kw in q for kw in keywords):
            return KnowledgeBoundaryResult(
                knowledge_type=KnowledgeType.KNOWN_KNOWN,
                confidence=conf,
                response_strategy='answer_directly',
                matched_signal=domain
            )
    return KnowledgeBoundaryResult(
        knowledge_type=KnowledgeType.UNCERTAIN,
        confidence=0.55,
        response_strategy='answer_with_caveat_and_suggest_verification',
        suggested_source='Xác nhận với phòng pháp chế'
    )

# --- Test ---
test_queries = [
    'Credit score tối thiểu để approve loan là bao nhiêu?',
    'Lãi suất cơ bản NHNN hiện tại là bao nhiêu?',
    'FinTech Corp có hỗ trợ vay mua xe ô tô không?',
    'Thông tư mới nhất về tín dụng tiêu dùng là gì?',
]

print('=== Knowledge Boundary Classification ===')
for q in test_queries:
    result = classify_knowledge_boundary(q)
    icon = {'known_known': '✅', 'known_unknown': '❌', 'uncertain': '⚠️'}[result.knowledge_type.value]
    print(f'{icon} [{result.knowledge_type.value.upper()}] conf={result.confidence:.2f} | {q[:60]}')
    print(f'   → Strategy: {result.response_strategy}')
    if result.suggested_source:
        print(f'   → Source: {result.suggested_source}')
    print()

## Section 2: Hallucination Detection — Citation Verifier + Numeric Bounds

In [ ]:
VALID_REGULATIONS = {
    'TT39/2016': {'articles': range(1, 28), 'full_name': 'Thông tư 39/2016/TT-NHNN', 'topic': 'Cho vay của tổ chức tín dụng'},
    'NĐ13/2023': {'articles': range(1, 52), 'full_name': 'Nghị định 13/2023/NĐ-CP', 'topic': 'Bảo vệ dữ liệu cá nhân'},
    'TT16/2021': {'articles': range(1, 23), 'full_name': 'Thông tư 16/2021/TT-NHNN', 'topic': 'Mua bán trái phiếu doanh nghiệp'},
    'TT12/2024': {'articles': range(1, 35), 'full_name': 'Thông tư 12/2024/TT-NHNN', 'topic': 'Sửa đổi TT39 về DTI threshold'},
}

NUMERIC_BOUNDS = {
    'confidence':       (0.0,   1.0),
    'credit_score':     (300,   900),
    'dti_ratio':        (0.0,   1.0),
    'interest_rate':    (0.01,  0.36),
    'loan_amount_vnd':  (10_000_000, 10_000_000_000),
    'employment_months':(0,     600),
}

def verify_citations(regulation_refs: List[str]) -> List[dict]:
    results = []
    for ref in regulation_refs:
        match = re.search(r'(TT\d+/\d{4}|NĐ\d+/\d{4})', ref)
        article_match = re.search(r'(?:Điều|Art\.?)\s*(\d+)', ref, re.IGNORECASE)
        if not match:
            results.append({'ref': ref, 'valid': False, 'reason': 'Cannot parse regulation identifier'})
            continue
        reg_id = match.group(1)
        if reg_id not in VALID_REGULATIONS:
            results.append({'ref': ref, 'valid': False, 'reason': f'Unknown regulation: {reg_id}'})
            continue
        reg = VALID_REGULATIONS[reg_id]
        if article_match:
            article_num = int(article_match.group(1))
            if article_num not in reg['articles']:
                results.append({'ref': ref, 'valid': False,
                                'reason': f'Article {article_num} out of range for {reg_id} (valid: 1–{max(reg["articles"])})',
                                'hallucination_type': 'citation'})
                continue
        results.append({'ref': ref, 'valid': True, 'reason': f'Valid: {reg["full_name"]} — {reg["topic"]}'})
    return results

def check_numeric_bounds(output: dict) -> List[dict]:
    results = []
    for field, (low, high) in NUMERIC_BOUNDS.items():
        if field not in output: continue
        val = output[field]
        ok = low <= val <= high
        results.append({
            'field': field, 'value': val, 'valid': ok,
            'reason': f'OK: {val} in [{low}, {high}]' if ok else f'OUT OF RANGE: {val} not in [{low}, {high}]',
            'hallucination_type': None if ok else 'numeric'
        })
    return results

# Test with a realistic hallucinated LoanBot output
hallucinated = {
    'decision': 'APPROVE',
    'confidence': 0.94,
    'credit_score': 720,
    'dti_ratio': 0.38,
    'interest_rate': 0.025,       # 2.5%/month → not hallucinated (30%/year within range)
    'loan_amount_vnd': 500_000_000,
    'regulation_refs': [
        'TT39/2016 Điều 9',       # Valid
        'NĐ13/2023 Điều 65',      # HALLUCINATION — max article is 51
        'TT99/2022 Điều 3',       # HALLUCINATION — TT99/2022 doesn't exist
    ]
}

print('=== Citation Verification ===')
for r in verify_citations(hallucinated['regulation_refs']):
    icon = '✅' if r['valid'] else '❌'
    print(f'{icon} {r["ref"]}: {r["reason"]}')

print('\n=== Numeric Bounds Check ===')
for r in check_numeric_bounds(hallucinated):
    icon = '✅' if r['valid'] else '❌'
    print(f'{icon} {r["field"]}: {r["reason"]}')

## Section 3: Self-Consistency Sampling

In [ ]:
import random

# Mock LoanBot model — deterministic based on credit score with slight randomness
class MockLoanBot:
    def __init__(self, seed=42):
        self.rng = random.Random(seed)

    def decide(self, application: dict, sample_idx: int = 0) -> dict:
        score = application.get('credit_score', 600)
        dti   = application.get('dti_ratio', 0.40)
        # Borderline cases have higher variance
        is_borderline = 580 <= score <= 680
        noise = self.rng.gauss(0, 0.08 if is_borderline else 0.03)
        adjusted_score = score + noise * 100
        if adjusted_score >= 660 and dti < 0.43:
            decision, conf = 'APPROVE', 0.75 + self.rng.uniform(0, 0.15)
        elif adjusted_score >= 580 or dti < 0.45:
            decision, conf = 'REVIEW', 0.60 + self.rng.uniform(0, 0.20)
        else:
            decision, conf = 'REJECT', 0.80 + self.rng.uniform(0, 0.10)
        return {'decision': decision, 'confidence': round(conf, 3)}

def self_consistency_check(model: MockLoanBot, application: dict, n_samples: int = 5) -> dict:
    decisions, confidences = [], []
    for i in range(n_samples):
        result = model.decide({**application, '_sample_idx': i}, sample_idx=i)
        decisions.append(result['decision'])
        confidences.append(result['confidence'])

    decision_counts = {}
    for d in decisions:
        decision_counts[d] = decision_counts.get(d, 0) + 1
    majority = max(decision_counts, key=decision_counts.get)
    agreement_rate = decision_counts[majority] / n_samples
    mean_conf = statistics.mean(confidences)
    calibrated = round(agreement_rate * 0.6 + mean_conf * 0.4, 3)

    return {
        'majority_decision': majority,
        'agreement_rate': round(agreement_rate, 3),
        'mean_confidence': round(mean_conf, 3),
        'calibrated_confidence': calibrated,
        'distribution': decision_counts,
        'all_decisions': decisions,
        'should_escalate': agreement_rate < 0.67 or calibrated < 0.75
    }

model = MockLoanBot(seed=99)
test_cases = [
    {'id': 'FC-2024-001', 'credit_score': 720, 'dti_ratio': 0.32, 'label': 'Clear approve'},
    {'id': 'FC-2024-002', 'credit_score': 638, 'dti_ratio': 0.41, 'label': 'Borderline'},
    {'id': 'FC-2024-003', 'credit_score': 420, 'dti_ratio': 0.62, 'label': 'Clear reject'},
    {'id': 'FC-2024-004', 'credit_score': 648, 'dti_ratio': 0.44, 'label': 'Very borderline'},
]

print('=== Self-Consistency Sampling (5 samples each) ===')
for tc in test_cases:
    result = self_consistency_check(model, tc, n_samples=5)
    escalate_flag = '🚨 ESCALATE' if result['should_escalate'] else '✅ DELIVER'
    print(f"\n[{tc['id']}] {tc['label']}")
    print(f"  Decisions: {result['all_decisions']}")
    print(f"  Majority: {result['majority_decision']} | Agreement: {result['agreement_rate']:.2f} | Mean conf: {result['mean_confidence']:.3f}")
    print(f"  Calibrated conf: {result['calibrated_confidence']:.3f} | {escalate_flag}")

## Section 4: Calibration & ECE Measurement

In [ ]:
# Simulate LoanBot predictions with stated confidences
# Each: (stated_confidence, is_correct)
MOCK_PREDICTIONS = [
    # Overconfident cases (common in LLMs)
    (0.95, True), (0.92, True), (0.90, False), (0.93, True), (0.91, False),
    (0.88, True), (0.87, True), (0.89, False), (0.86, True), (0.90, True),
    # Mid-confidence cases
    (0.75, True), (0.72, False), (0.78, True), (0.70, True), (0.73, False),
    (0.76, True), (0.74, False), (0.77, True), (0.71, True), (0.79, False),
    # Low confidence cases
    (0.55, True), (0.58, False), (0.52, False), (0.60, True), (0.56, False),
    (0.53, True), (0.57, False), (0.59, True), (0.54, False), (0.61, True),
]

def compute_ece(predictions, n_bins=5):
    bins = {i: {'total': 0, 'correct': 0, 'conf_sum': 0} for i in range(n_bins)}
    bin_width = 1.0 / n_bins
    for conf, correct in predictions:
        bin_idx = min(int(conf / bin_width), n_bins - 1)
        bins[bin_idx]['total'] += 1
        bins[bin_idx]['conf_sum'] += conf
        if correct: bins[bin_idx]['correct'] += 1

    ece = 0.0
    print(f"{'Bin':>8} | {'Count':>5} | {'Avg Conf':>8} | {'Accuracy':>8} | {'|Δ|':>6}")
    print('-' * 50)
    for i, b in bins.items():
        if b['total'] == 0: continue
        avg_conf = b['conf_sum'] / b['total']
        accuracy = b['correct'] / b['total']
        delta = abs(avg_conf - accuracy)
        ece += delta * b['total'] / len(predictions)
        lo = i * bin_width
        print(f"{lo:.1f}-{lo+bin_width:.1f}  | {b['total']:>5} | {avg_conf:>8.3f} | {accuracy:>8.3f} | {delta:>6.3f}")
    return ece

print('=== Calibration Analysis (ECE) ===')
ece = compute_ece(MOCK_PREDICTIONS)
print(f'\nECE = {ece:.4f}')
if ece < 0.05:
    print('✅ Well calibrated — model confidence tracks reality')
elif ece < 0.10:
    print('⚠️ Acceptable calibration — monitor closely')
else:
    print('❌ Poor calibration — apply temperature scaling or Platt scaling')

# Diagnose overconfidence vs underconfidence
high_conf = [(c, ok) for c, ok in MOCK_PREDICTIONS if c >= 0.85]
high_conf_accuracy = sum(1 for _, ok in high_conf if ok) / len(high_conf)
avg_high_conf = statistics.mean(c for c, _ in high_conf)
print(f'\nHigh-confidence cases (conf≥0.85): stated={avg_high_conf:.3f}, actual accuracy={high_conf_accuracy:.3f}')
if avg_high_conf > high_conf_accuracy:
    print(f'→ OVERCONFIDENT by {avg_high_conf - high_conf_accuracy:.3f} — typical LLM pattern')

## Section 5: RAG Faithfulness Checker

In [ ]:
@dataclass
class RetrievedChunk:
    text: str
    source: str
    relevance_score: float

@dataclass
class FaithfulnessResult:
    faithfulness_score: float  # 0–1: fraction of claims grounded in context
    relevance_score: float     # 0–1: avg relevance of retrieved chunks
    grounded_claims: List[str]
    ungrounded_claims: List[str]
    verdict: str  # PASS / WARN / FAIL

def check_faithfulness(generated_text: str, chunks: List[RetrievedChunk]) -> FaithfulnessResult:
    """Simplified faithfulness: check if key claims appear in retrieved context."""
    # Extract claims as sentences
    sentences = [s.strip() for s in generated_text.split('.') if len(s.strip()) > 10]
    context_text = ' '.join(c.text.lower() for c in chunks)

    grounded, ungrounded = [], []
    for sent in sentences:
        # Check if key phrases from sentence appear in context
        words = [w for w in sent.lower().split() if len(w) > 4]
        if not words: continue
        overlap_count = sum(1 for w in words if w in context_text)
        overlap_ratio = overlap_count / len(words) if words else 0
        if overlap_ratio >= 0.35:  # at least 35% word overlap
            grounded.append(sent)
        else:
            ungrounded.append(sent)

    total = len(grounded) + len(ungrounded)
    faithfulness = len(grounded) / total if total > 0 else 0
    relevance = statistics.mean(c.relevance_score for c in chunks) if chunks else 0

    if faithfulness >= 0.90 and relevance >= 0.80:
        verdict = 'PASS'
    elif faithfulness >= 0.70:
        verdict = 'WARN'
    else:
        verdict = 'FAIL — likely hallucination'

    return FaithfulnessResult(faithfulness, relevance, grounded, ungrounded, verdict)

# Test scenarios
good_chunks = [
    RetrievedChunk('Điều 9 TT39/2016 quy định điều kiện vay vốn: mục đích vay hợp pháp, có khả năng tài chính, hoàn trả đúng hạn.', 'TT39/2016 Điều 9', 0.92),
    RetrievedChunk('Khách hàng cần cung cấp giấy tờ tùy thân, hợp đồng lao động, sao kê tài khoản 3 tháng gần nhất.', 'TT39/2016 Điều 11', 0.88),
]
bad_chunks = [  # low-relevance, off-topic
    RetrievedChunk('Trái phiếu doanh nghiệp được mua bán theo Thông tư 16/2021/TT-NHNN.', 'TT16/2021 Điều 1', 0.30),
]

grounded_response = 'Theo TT39/2016 Điều 9, điều kiện vay vốn gồm mục đích hợp pháp và khả năng tài chính. Khách hàng cần cung cấp hợp đồng lao động và sao kê tài khoản.'
hallucinated_response = 'Theo TT39/2016 Điều 9, khách hàng VIP với credit score trên 750 được giảm lãi suất 0.5%/năm. Chương trình ưu đãi áp dụng đến hết quý 4/2024.'

print('=== Faithfulness Check: Grounded Response ===')
r1 = check_faithfulness(grounded_response, good_chunks)
print(f'Faithfulness: {r1.faithfulness_score:.2f} | Relevance: {r1.relevance_score:.2f} | Verdict: {r1.verdict}')

print('\n=== Faithfulness Check: Hallucinated Response (bad chunks) ===')
r2 = check_faithfulness(hallucinated_response, bad_chunks)
print(f'Faithfulness: {r2.faithfulness_score:.2f} | Relevance: {r2.relevance_score:.2f} | Verdict: {r2.verdict}')
if r2.ungrounded_claims:
    print('Ungrounded claims:')
    for claim in r2.ungrounded_claims:
        print(f'  ❌ "{claim}"')

## Section 6: Knowledge Freshness Monitor

In [ ]:
@dataclass
class KnowledgeSource:
    name: str
    last_updated: date
    update_frequency_days: int
    staleness_threshold_days: int
    source_url: str

KNOWLEDGE_SOURCES = [
    KnowledgeSource('TT39/2016',             date(2024,  1, 15), 365,  90,  'sbv.gov.vn'),
    KnowledgeSource('NĐ13/2023',             date(2024,  1, 15), 365,  90,  'vanban.chinhphu.vn'),
    KnowledgeSource('TT16/2021',             date(2024,  1, 15), 365,  90,  'sbv.gov.vn'),
    KnowledgeSource('TT12/2024',             date(2024,  8,  1),  90,  30,  'sbv.gov.vn'),
    KnowledgeSource('FinTech Internal Policy', date(2024, 3,  1),  30,  45,  'internal'),
    KnowledgeSource('NHNN Interest Rate',    date(2024,  2,  1),   7,  14,  'sbv.gov.vn/rate'),
]

def check_knowledge_freshness(as_of: date = None) -> List[dict]:
    today = as_of or date.today()
    report = []
    for src in KNOWLEDGE_SOURCES:
        days = (today - src.last_updated).days
        is_stale = days > src.staleness_threshold_days
        if days > src.staleness_threshold_days * 2:
            severity = 'HIGH'
        elif is_stale:
            severity = 'MEDIUM'
        else:
            severity = 'OK'
        report.append({
            'source': src.name, 'days_since_update': days,
            'stale': is_stale, 'severity': severity,
            'source_url': src.source_url,
            'update_action': f'Check {src.source_url}' if is_stale else 'No action needed'
        })
    return report

def temporal_hedge(response: dict, topic: str) -> dict:
    TIME_SENSITIVE = ['lãi suất', 'interest rate', 'policy threshold', 'thông tư mới', 'dti limit']
    if any(t in topic.lower() for t in TIME_SENSITIVE):
        response['temporal_caveat'] = (
            f'⚠️ Thông tin về "{topic}" dựa trên knowledge của model có thể đã lỗi thời. '
            f'Vui lòng xác nhận tại nguồn chính thức trước khi đưa ra quyết định.'
        )
        response['requires_verification'] = True
    return response

# Simulate: checking freshness as of 2024-11-01
check_date = date(2024, 11, 1)
freshness_report = check_knowledge_freshness(as_of=check_date)

print(f'=== Knowledge Freshness Report (as of {check_date}) ===')
severity_icons = {'OK': '✅', 'MEDIUM': '⚠️', 'HIGH': '🚨'}
for item in sorted(freshness_report, key=lambda x: x['days_since_update'], reverse=True):
    icon = severity_icons[item['severity']]
    print(f"{icon} [{item['severity']:6}] {item['source']:30} — {item['days_since_update']:3d} days stale")
    if item['stale']:
        print(f"   → Action: {item['update_action']}")

stale_count = sum(1 for r in freshness_report if r['stale'])
print(f'\nSummary: {stale_count}/{len(freshness_report)} sources require update')

## Section 7: Epistemic Guard Orchestrator

Full pipeline kết hợp tất cả components.

**TODO:** Implement `EpistemicGuard.evaluate()` hoàn chỉnh.
- Bước 1: Classify knowledge boundary của query
- Bước 2: Nếu KNOWN_UNKNOWN → decline và redirect  
- Bước 3: Nếu UNCERTAIN hoặc KNOWN_KNOWN → run model, verify citations và numerics
- Bước 4: Self-consistency sampling (n=3)
- Bước 5: Check calibrated confidence threshold
- Bước 6: Apply temporal hedge nếu time-sensitive topic
- Bước 7: Return EpistemicGuardResult

In [ ]:
@dataclass
class LoanQuery:
    query_text: str
    topic: str
    application: Optional[dict] = None

@dataclass
class EpistemicGuardResult:
    should_deliver: bool
    action: str  # 'deliver', 'deliver_with_caveat', 'escalate', 'decline_and_redirect'
    decision: Optional[str] = None
    calibrated_confidence: Optional[float] = None
    hallucination_flags: List[str] = field(default_factory=list)
    temporal_caveat: Optional[str] = None
    redirect_source: Optional[str] = None
    explanation: str = ''

class EpistemicGuard:
    def __init__(self, model: MockLoanBot, n_samples: int = 3):
        self.model = model
        self.n_samples = n_samples

    def evaluate(self, query: LoanQuery) -> EpistemicGuardResult:
        """TODO: Implement full epistemic guard pipeline."""
        raise NotImplementedError('Implement this method!')

class EpistemicGuardSolution(EpistemicGuard):
    def evaluate(self, query: LoanQuery) -> EpistemicGuardResult:
        # Step 1: Knowledge boundary
        boundary = classify_knowledge_boundary(query.query_text)

        # Step 2: Decline if out-of-domain
        if boundary.knowledge_type == KnowledgeType.KNOWN_UNKNOWN:
            return EpistemicGuardResult(
                should_deliver=False, action='decline_and_redirect',
                redirect_source=boundary.suggested_source,
                explanation=f'Query outside knowledge boundary (signal: {boundary.matched_signal}). Redirect to {boundary.suggested_source}.'
            )

        # Step 3: Run model if application provided
        if query.application is None:
            return EpistemicGuardResult(should_deliver=True, action='deliver', explanation='No application provided — boundary check only')

        mock_output = self.model.decide(query.application)
        hallucination_flags = []

        # Citation and numeric checks on mock output
        if 'regulation_refs' in mock_output:
            for r in verify_citations(mock_output.get('regulation_refs', [])):
                if not r['valid']:
                    hallucination_flags.append(f"Citation: {r['ref']} — {r['reason']}")
        for r in check_numeric_bounds(mock_output):
            if not r['valid']:
                hallucination_flags.append(f"Numeric: {r['field']}={r['value']} — {r['reason']}")

        # Step 4: Self-consistency sampling
        sc = self_consistency_check(self.model, query.application, n_samples=self.n_samples)

        # Step 5: Confidence threshold
        result_base = EpistemicGuardResult(
            should_deliver=not sc['should_escalate'] and len(hallucination_flags) == 0,
            decision=sc['majority_decision'],
            calibrated_confidence=sc['calibrated_confidence'],
            hallucination_flags=hallucination_flags,
            action='',
        )
        if hallucination_flags:
            result_base.action = 'quarantine'
            result_base.explanation = f'Hallucination detected: {len(hallucination_flags)} issues'
        elif sc['should_escalate']:
            result_base.action = 'escalate'
            result_base.explanation = f'Low calibrated confidence ({sc["calibrated_confidence"]:.3f}) — human review required'
        else:
            result_base.action = 'deliver'
            result_base.explanation = f'All checks passed. Calibrated conf: {sc["calibrated_confidence"]:.3f}'

        # Step 6: Temporal hedge
        TIME_SENSITIVE = ['lãi suất', 'interest rate', 'dti limit', 'threshold']
        if any(t in query.topic.lower() for t in TIME_SENSITIVE):
            result_base.temporal_caveat = f'⚠️ Thông tin về "{query.topic}" cần xác nhận từ nguồn chính thức.'
            if result_base.action == 'deliver':
                result_base.action = 'deliver_with_caveat'

        return result_base

# --- Test ---
guard = EpistemicGuardSolution(model=MockLoanBot(seed=42), n_samples=5)

test_queries = [
    LoanQuery('Lãi suất cơ bản hiện tại là bao nhiêu?', topic='lãi suất'),
    LoanQuery('Hồ sơ FC-2024-001 có đủ điều kiện vay không?', topic='loan decision', application={'credit_score': 720, 'dti_ratio': 0.32}),
    LoanQuery('Hồ sơ FC-2024-002 có đủ điều kiện vay không?', topic='loan decision', application={'credit_score': 638, 'dti_ratio': 0.41}),
]

print('=== Epistemic Guard Orchestrator ===')
for q in test_queries:
    result = guard.evaluate(q)
    action_icons = {'deliver': '✅', 'deliver_with_caveat': '⚠️✅', 'escalate': '🚨', 'quarantine': '🔒', 'decline_and_redirect': '↩️'}
    print(f"\nQuery: {q.query_text}")
    print(f"Action: {action_icons.get(result.action, '?')} {result.action.upper()}")
    if result.decision: print(f"Decision: {result.decision} (conf: {result.calibrated_confidence:.3f})")
    if result.redirect_source: print(f"Redirect to: {result.redirect_source}")
    if result.temporal_caveat: print(f"Caveat: {result.temporal_caveat}")
    if result.hallucination_flags: print(f"Hallucinations: {result.hallucination_flags}")
    print(f"Explanation: {result.explanation}")